## RAPTOR：递归摘要树（Recursive Abstractive Processing for Tree-Organized Retrieval）

RAPTOR 的核心想法是：

- **离线索引阶段（Indexing）**：把长文档（或多文档）切成基础单元（这里用“页面文本”演示），对这些文本做向量表示，然后用聚类把“语义相近的一组”聚在一起；对每个簇生成一个**摘要节点**。摘要节点再继续聚类、再摘要……最终形成一棵“从细节到概括”的**摘要树**。
- **在线查询阶段（Querying）**：先在更高层（更抽象）的摘要节点里找与问题最相关的主题，再沿着树向下取对应的更细粒度内容，最后把这些内容组织成上下文交给模型回答。

这种结构让系统能够在“全局概念”和“局部细节”之间切换，而不是只在同一粒度的 chunks 里做 Top-K。

<div style="text-align: center;">
<img src="../images/raptor.svg" alt="RAPTOR" style="width:100%; height:auto;" />
</div>

## 环境准备

请确保当前环境已安装：

- `langchain-core`, `langchain-openai`, `langchain-community`
- `faiss-cpu`
- `pypdf`
- `numpy`, `pandas`, `scikit-learn`, `matplotlib`
- `python-dotenv`, `requests`

In [1]:
import os
import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from sklearn.mixture import GaussianMixture

import pypdf

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import os
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
os.environ["ALL_PROXY"] = "http://127.0.0.1:7890"

## 准备示例文档（PDF）

为了把流程讲清楚，我们沿用原仓库的示例文档：`Understanding_Climate_Change.pdf`。

- 我们会把 **每一页**当作 level-0 的“原始节点”。
- 为了控制成本，先用 `MAX_PAGES` 截取前几页跑通流程。

In [3]:
def download_file(url: str, dst: Path, *, timeout_s: float = 60.0) -> Path:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 0:
        return dst

    r = requests.get(url, timeout=timeout_s)
    r.raise_for_status()
    dst.write_bytes(r.content)
    return dst


pdf_url = "https://raw.githubusercontent.com/NirDiamant/RAG_Techniques/main/data/Understanding_Climate_Change.pdf"
pdf_path = download_file(pdf_url, DATA_DIR / "Understanding_Climate_Change.pdf")

str(pdf_path)

'../data/Understanding_Climate_Change.pdf'

In [4]:
def load_pdf_pages(file_path: Path, *, max_pages: int | None = None) -> list[Document]:
    reader = pypdf.PdfReader(str(file_path))
    pages = reader.pages
    if max_pages is not None:
        pages = pages[:max_pages]

    docs: list[Document] = []
    for i, page in enumerate(pages):
        text = page.extract_text() or ""
        docs.append(Document(page_content=text, metadata={"source": str(file_path), "page": i}))
    return docs


MAX_PAGES = 12  # 先小规模跑通流程，确认没问题再增大
page_docs = load_pdf_pages(pdf_path, max_pages=MAX_PAGES)
texts_level0 = [d.page_content for d in page_docs]

len(texts_level0), texts_level0[0][:200]

(12,
 'Understanding Climate Change \nChapter 1: Introduction to Climate Change \nClimate change refers to significant, long-term changes in the global climate. The term \n"global climate" encompasses the plane')

## 定义 embeddings 与 LLM

RAPTOR 需要两类模型能力：

- **Embeddings**：把每个节点（原文页/摘要）编码成向量，用于聚类与相似检索
- **LLM**：对每个簇生成摘要（“抽象上卷”），以及最终基于检索到的上下文回答问题

这里用 OpenAI 作为示例，你也可以替换成其它 provider。

In [5]:
EMBEDDING_MODEL = os.getenv("RAPTOR_EMBEDDING_MODEL", "text-embedding-3-large")
LLM_MODEL = os.getenv("RAPTOR_LLM_MODEL", "gpt-4o-mini")

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

EMBEDDING_MODEL, LLM_MODEL

2026-06-20 16:15:26,148 - INFO - langchain-openai detected HTTP_PROXY, HTTPS_PROXY, ALL_PROXY and no explicit `http_socket_options` / `http_client` / `http_async_client` / `openai_proxy`; skipping the custom `httpx` transport so httpx's env-proxy auto-detection applies. Pass `http_socket_options=[...]` to opt back into kernel-level TCP keepalive tuning on top of the env proxy.


('text-embedding-3-large', 'gpt-4o-mini')

## RAPTOR：建树（Indexing / Offline）

我们会实现 4 个关键动作：

- **embed_texts**：把一批文本转成向量
- **perform_clustering**：对向量做聚类（这里用 `GaussianMixture`）
- **summarize_cluster**：对每个簇的文本生成“摘要节点”
- **build_raptor_tree**：重复上述过程，把 level-0 原文逐层“卷”成 level-1/2/… 的摘要

输出是一个 `dict[level, DataFrame]`，每层都包含：

- `id`：节点 ID
- `text`：节点文本（原文页或摘要）
- `vector`：该文本的向量
- `cluster`：本层聚类簇 ID
- `metadata`：`level/origin/child_ids/...`

In [6]:
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def embed_texts(texts: list[str]) -> np.ndarray:
    vecs = embeddings.embed_documents(texts)
    return np.array(vecs, dtype=np.float32)


def perform_clustering(vectors: np.ndarray, *, max_clusters: int = 10, seed: int = 42) -> np.ndarray:
    n = int(vectors.shape[0])
    if n <= 1:
        return np.zeros((n,), dtype=np.int64)

    n_clusters = max(1, min(max_clusters, n // 2))
    if n_clusters == 1:
        return np.zeros((n,), dtype=np.int64)

    gm = GaussianMixture(n_components=n_clusters, random_state=seed)
    labels = gm.fit_predict(vectors)
    return labels.astype(np.int64)


summarize_prompt = ChatPromptTemplate.from_template(
    """请把下面这些材料压缩成一个更高层次的摘要，要求：

- 只保留关键事实与核心概念
- 尽量覆盖材料中的不同子主题
- 控制在 8~12 句左右

材料：
{text}
"""
)

summarize_chain = summarize_prompt | llm | StrOutputParser()


def summarize_cluster(texts: list[str]) -> str:
    joined = "\n\n".join(t.strip() for t in texts if t and t.strip())
    if not joined:
        return ""
    return summarize_chain.invoke({"text": joined})


def build_raptor_tree(
    texts_level0: list[str],
    *,
    max_levels: int = 3,
    max_clusters: int = 10,
) -> dict[int, pd.DataFrame]:
    """从 level-0 原始文本开始，递归聚类+摘要，构建多层摘要树。"""

    results: dict[int, pd.DataFrame] = {}

    # level-0 节点
    current_texts = [t for t in texts_level0]
    current_ids = [f"L0_{i:04d}" for i in range(len(current_texts))]
    current_metadata = [
        {
            "id": node_id,
            "level": 0,
            "origin": "original_page",
            "child_ids": [],
        }
        for node_id in current_ids
    ]

    for level in range(0, max_levels + 1):
        logging.info("Embedding level=%s nodes=%s", level, len(current_texts))
        vectors = embed_texts(current_texts)

        labels = perform_clustering(vectors, max_clusters=max_clusters)

        df = pd.DataFrame(
            {
                "id": current_ids,
                "text": current_texts,
                "vector": list(vectors),
                "cluster": labels,
                "metadata": current_metadata,
            }
        )
        results[level] = df

        # 停止条件：下一层没有意义
        if level >= max_levels or len(current_texts) <= 1:
            break

        # 生成下一层摘要节点
        next_texts: list[str] = []
        next_ids: list[str] = []
        next_metadata: list[dict[str, Any]] = []

        for cluster_id in sorted(df["cluster"].unique().tolist()):
            cluster_df = df[df["cluster"] == cluster_id]
            child_ids = cluster_df["id"].tolist()
            cluster_texts = cluster_df["text"].tolist()

            summary_text = summarize_cluster(cluster_texts)
            summary_id = f"L{level+1}_C{int(cluster_id):02d}"

            next_texts.append(summary_text)
            next_ids.append(summary_id)
            next_metadata.append(
                {
                    "id": summary_id,
                    "level": level + 1,
                    "origin": f"summary_of_level_{level}_cluster_{int(cluster_id)}",
                    "child_ids": child_ids,
                }
            )

        current_texts = next_texts
        current_ids = next_ids
        current_metadata = next_metadata

    return results

### 构建摘要树

运行后你会得到 `tree_results[level]`：

- `level=0`：原始页面节点
- `level>=1`：每一层聚类后生成的“摘要节点”

你可以把它理解为：**越高层越抽象、越低层越具体**。

In [7]:
MAX_LEVELS = 3
MAX_CLUSTERS = 10

tree_results = build_raptor_tree(texts_level0, max_levels=MAX_LEVELS, max_clusters=MAX_CLUSTERS)

{k: len(v) for k, v in tree_results.items()}

2026-06-20 16:15:41,981 - INFO - Embedding level=0 nodes=12
2026-06-20 16:15:43,699 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/embeddings "HTTP/1.1 200 OK"
2026-06-20 16:15:53,672 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-20 16:15:56,634 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-20 16:16:00,413 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-20 16:16:03,783 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-20 16:16:11,808 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-20 16:16:14,450 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-20 16:16:14,455 - INFO - Embedding level=1 nodes=6
2026-06-20 16:16:15,583 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/em

{0: 12, 1: 6, 2: 3, 3: 1}

In [8]:
# 观察一下最高层摘要（如果存在）
max_level = max(tree_results.keys())
top_df = tree_results[max_level]

max_level, top_df[['id','text']].head(3).to_dict(orient='records')

(3,
 [{'id': 'L3_C00',
   'text': '气候变化是由人类活动引起的全球性问题，主要源于化石燃料的燃烧和森林砍伐，导致温室气体（如二氧化碳、甲烷和氧化亚氮）排放增加。农业、交通和电力生产是主要的排放源，而森林的破坏则削弱了碳汇的作用。气候变化引发全球气温上升、极端天气频发，影响人类健康、农业和生态系统。应对气候变化需要综合策略，包括提高能源效率、推广可再生能源和实施有效的气候政策，如《巴黎协定》。植树造林和可持续农业实践对增强生态健康和气候韧性至关重要。新兴技术（如人工智能和碳捕集技术）为减排提供了创新解决方案，而地方政府和社区的参与也是推动气候行动的重要力量。全球合作和持续的研究创新是应对气候变化的关键。'}])

## 在线查询：层级检索 + 上下文压缩（Query / Online）

查询阶段我们做 3 件事：

1. **从最高层开始检索**：先找“最相关的摘要节点”，确定问题属于哪个主题簇
2. **沿树向下取子节点**：把这些摘要节点的 `child_ids` 当作下一层候选，再在候选里检索更细粒度内容
3. **上下文压缩（可选）**：对检索到的文本做一次“只抽取和问题相关的片段”，减少噪声与 token 消耗

> 这里为了让机制清晰，我们用“向量 + 元数据（level、child_ids）”在内存里做层级检索；它和“把全部节点塞进一个向量库，然后在查询时做过滤/二次检索”本质一致。

In [9]:
@dataclass(frozen=True)
class Node:
    id: str
    text: str
    vector: np.ndarray
    metadata: dict[str, Any]


def build_node_index(tree_results: dict[int, pd.DataFrame]) -> tuple[dict[int, list[Node]], dict[int, dict[str, Node]]]:
    nodes_by_level: dict[int, list[Node]] = {}
    by_id: dict[int, dict[str, Node]] = {}

    for level, df in tree_results.items():
        level_nodes: list[Node] = []
        level_map: dict[str, Node] = {}
        for _, r in df.iterrows():
            node = Node(
                id=str(r["id"]),
                text=str(r["text"]),
                vector=np.array(r["vector"], dtype=np.float32),
                metadata=dict(r["metadata"]),
            )
            level_nodes.append(node)
            level_map[node.id] = node
        nodes_by_level[level] = level_nodes
        by_id[level] = level_map

    return nodes_by_level, by_id


nodes_by_level, nodes_by_id = build_node_index(tree_results)
max(nodes_by_level.keys()), [len(v) for _, v in sorted(nodes_by_level.items())]

(3, [12, 6, 3, 1])

In [10]:
compress_prompt = ChatPromptTemplate.from_template(
    """你会看到一段上下文和一个问题。
请只抽取对回答问题必需的信息（可以是要点列表），不要复述无关内容。

问题：{question}

上下文：
{context}

相关信息：
"""
)

compress_chain = compress_prompt | llm | StrOutputParser()


def compress_doc(doc: Document, *, question: str) -> Document:
    compressed = compress_chain.invoke({"question": question, "context": doc.page_content})
    meta = dict(doc.metadata)
    meta["compressed"] = True
    return Document(page_content=compressed, metadata=meta)


def topk_by_cosine(
    nodes: list[Node],
    *,
    query_vec: np.ndarray,
    k: int,
    restrict_ids: set[str] | None = None,
) -> list[tuple[Node, float]]:
    scored: list[tuple[Node, float]] = []
    for n in nodes:
        if restrict_ids is not None and n.id not in restrict_ids:
            continue
        s = cosine_sim(query_vec, n.vector)
        scored.append((n, s))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]


def raptor_retrieve(
    question: str,
    *,
    k_per_level: int = 3,
    do_compress: bool = True,
) -> list[Document]:
    qv = np.array(embeddings.embed_query(question), dtype=np.float32)

    retrieved: list[Document] = []
    restrict: set[str] | None = None

    for level in range(max(nodes_by_level.keys()), -1, -1):
        scored = topk_by_cosine(nodes_by_level[level], query_vec=qv, k=k_per_level, restrict_ids=restrict)

        docs_level: list[Document] = []
        next_restrict: set[str] = set()

        for node, score in scored:
            meta = dict(node.metadata)
            meta["score"] = float(score)
            docs_level.append(Document(page_content=node.text, metadata=meta))
            next_restrict.update(meta.get("child_ids", []) or [])

        if do_compress:
            docs_level = [compress_doc(d, question=question) for d in docs_level]

        retrieved.extend(docs_level)
        restrict = next_restrict if next_restrict else None

    return retrieved

In [11]:
answer_prompt = ChatPromptTemplate.from_template(
    """你是一个严谨的问答助手。
请只基于给定的上下文回答问题；如果上下文不足以回答，就说“我不知道”。

问题：{question}

上下文：
{context}

回答：
"""
)

answer_chain = answer_prompt | llm | StrOutputParser()


def raptor_answer(question: str, *, k_per_level: int = 3, do_compress: bool = True) -> dict[str, Any]:
    docs = raptor_retrieve(question, k_per_level=k_per_level, do_compress=do_compress)
    context = "\n\n".join(
        f"[L{d.metadata.get('level')}, score={d.metadata.get('score'):.3f}]\n{d.page_content}" for d in docs
    )
    answer = answer_chain.invoke({"question": question, "context": context})

    return {
        "question": question,
        "answer": answer,
        "docs": docs,
        "context": context,
    }


question = "What is the greenhouse effect?"
result = raptor_answer(question, k_per_level=2, do_compress=True)

result["answer"]

2026-06-20 16:16:41,488 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/embeddings "HTTP/1.1 200 OK"
2026-06-20 16:16:43,576 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-20 16:16:45,558 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-20 16:16:47,387 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-20 16:16:49,205 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-20 16:16:50,368 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-20 16:16:51,344 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-20 16:16:52,782 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-20 16:16:55,061 - INFO - HTTP Request: POST https://new.gptgod.cloud/v1/chat/c

'温室效应是指温室气体（如二氧化碳、甲烷和氧化亚氮）在大气中吸收和再辐射热量，导致地球表面温度上升的现象。该效应是自然现象，但人类活动（如燃烧化石燃料、森林砍伐）加剧了温室气体的排放，增强了温室效应，结果导致全球变暖和气候变化，影响生态系统和人类生活。'

In [12]:
def print_query_details(res: dict[str, Any], *, max_chars: int = 400):
    print("Question:\n", res["question"])
    print("\nAnswer:\n", res["answer"])

    print("\nRetrieved docs (by level):")
    for i, d in enumerate(res["docs"], 1):
        level = d.metadata.get("level")
        origin = d.metadata.get("origin")
        score = d.metadata.get("score")
        node_id = d.metadata.get("id")
        print(f"\n[{i}] level={level} id={node_id} score={score:.4f} origin={origin}")
        print(d.page_content[:max_chars].replace("\n", " ") + ("..." if len(d.page_content) > max_chars else ""))


print_query_details(result)

Question:
 What is the greenhouse effect?

Answer:
 温室效应是指温室气体（如二氧化碳、甲烷和氧化亚氮）在大气中吸收和再辐射热量，导致地球表面温度上升的现象。该效应是自然现象，但人类活动（如燃烧化石燃料、森林砍伐）加剧了温室气体的排放，增强了温室效应，结果导致全球变暖和气候变化，影响生态系统和人类生活。

Retrieved docs (by level):

[1] level=3 id=L3_C00 score=0.2971 origin=summary_of_level_2_cluster_0
- 温室气体（如二氧化碳、甲烷和氧化亚氮）排放增加 - 主要源于化石燃料的燃烧和森林砍伐 - 农业、交通和电力生产是主要的排放源 - 森林的破坏削弱了碳汇的作用 - 导致全球气温上升和极端天气频发 - 影响人类健康、农业和生态系统

[2] level=2 id=L2_C00 score=0.3202 origin=summary_of_level_1_cluster_0
- 温室效应是由温室气体（如二氧化碳、甲烷和氧化亚氮）引起的。 - 主要来源包括化石燃料的燃烧（煤炭、石油、天然气）和森林砍伐。 - 森林的碳汇作用被削弱，导致碳释放。 - 农业活动（牲畜、稻田、合成肥料）也加剧温室气体排放。 - 结果是全球气温上升和极端天气事件频发。

[3] level=2 id=L2_C01 score=0.2300 origin=summary_of_level_1_cluster_1
- 温室效应是指温室气体（如二氧化碳、甲烷等）在大气中吸收和再辐射热量，导致地球表面温度上升的现象。 - 该效应是自然现象，但人类活动（如燃烧化石燃料、森林砍伐）加剧了温室气体的排放，增强了温室效应。 - 结果是全球变暖和气候变化，影响生态系统和人类生活。

[4] level=1 id=L1_C02 score=0.3216 origin=summary_of_level_0_cluster_2
- 温室效应是指温室气体（如二氧化碳、甲烷和氧化亚氮）增强地球的温室效应，导致气候变暖。 - 温室气体的增加主要来源于化石燃料的燃烧。 - 现代气候变化主要由人类活动引起，尤其是自工业革命以来化石燃料消费的显著上升。

[5] level=1 id=